# Data preprocessing

Turns the raw EM-DAT CSVs into the small JSON files the D3 charts load.

**Inputs:** `preprocessing/raw/*.csv` (committed to the repo so this notebook runs offline)

**Outputs:** `data/*.json`, one file per chart, plus `sources.json` with the citations shown in the site's methodology section.

The raw files were downloaded from the Our World in Data API. To refresh them, run the commands in the README under *Reproducing the data*, then re-run this notebook top to bottom.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAW = Path("raw")
OUT = Path("..") / "data"
OUT.mkdir(exist_ok=True)

# The decade containing this year is incomplete, and every output flags it
# so the charts can label it "partial" instead of letting it look like a drop.
CURRENT_YEAR = 2026

TYPE_COLUMNS = [
    "Droughts", "Earthquakes", "Volcanoes", "Floods", "Landslides",
    "Storms", "Wildfires", "Extreme temperatures",
]


def decade_of(year):
    return (year // 10) * 10


def is_partial_decade(decade):
    return decade_of(CURRENT_YEAR) == decade


def load(name):
    return pd.read_csv(RAW / f"{name}.csv")


def load_metadata(name):
    with open(RAW / f"{name}.metadata.json", encoding="utf-8") as f:
        return json.load(f)


def write_json(name, payload):
    with open(OUT / name, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, separators=(",", ":"))
    print(f"wrote data/{name}")

## 1. Recorded disaster events per decade (step 1)

Yearly event counts grouped into decades. The source ships two derived
"excluding X" series that repeat the same events minus one category, so a
stacked chart would double count them; they are dropped.

In [2]:
df = load("number-of-natural-disaster-events")
df = df[~df["Entity"].str.startswith("All disasters excluding")]
df["Decade"] = df["Year"].apply(decade_of)

grouped = df.groupby(["Decade", "Entity"], as_index=False)["Disasters"].sum()
decades = sorted(grouped["Decade"].unique().tolist())

series = {}
for entity in grouped["Entity"].unique():
    sub = grouped[grouped["Entity"] == entity].set_index("Decade")["Disasters"]
    series[entity] = [int(sub.get(d, 0)) for d in decades]

write_json("events_by_decade.json", {
    "decades": decades,
    "partial_decade": decades[-1] if is_partial_decade(decades[-1]) else None,
    "series": series,
})

wrote data/events_by_decade.json


## 2. World death rate per 100,000 people (step 2)

The source already provides decade averages, so this is a straight extract
of the World rows.

In [3]:
df = load("decadal-average-death-rates-from-natural-disasters")
world = df[df["Entity"] == "World"].sort_values("Year")

decades = world["Year"].tolist()
write_json("death_rate_world.json", {
    "decades": decades,
    "partial_decade": decades[-1] if is_partial_decade(decades[-1]) else None,
    "by_type": {col: world[col].round(4).tolist() for col in TYPE_COLUMNS},
    "total": world["All disasters"].round(4).tolist(),
})

wrote data/death_rate_world.json


## 3. Absolute deaths per decade by disaster type (steps 3, 4 and 5)

Yearly deaths summed into decades. `min_count=1` keeps a decade's sum as
missing when every year in it is missing, instead of turning it into zero;
the export then writes those as 0 explicitly, which is correct here because
EM-DAT records no deaths rather than unknown deaths for these gaps.

In [4]:
df = load("natural-disasters-deaths")
world = df[df["Entity"] == "World"].copy()
world["Decade"] = world["Year"].apply(decade_of)

grouped = world.groupby("Decade")[TYPE_COLUMNS].sum(min_count=1)
decades = grouped.index.tolist()

write_json("deaths_by_type_decade.json", {
    "decades": decades,
    "partial_decade": decades[-1] if is_partial_decade(decades[-1]) else None,
    "by_type": {
        col: [int(v) if pd.notna(v) else 0 for v in grouped[col]]
        for col in TYPE_COLUMNS
    },
})

wrote data/deaths_by_type_decade.json


## 4. Per-country death rates for the map (step 6)

The world-atlas topojson identifies countries by ISO 3166 numeric code,
while the source data carries alpha-3 codes, so the rows are joined through
a published crosswalk. Aggregate rows (World, continents, income groups)
have `OWID_*` pseudo-codes and are dropped; the world-level charts cover
them instead. Missing rates stay `null` and render as "no data".

In [5]:
df = load("decadal-average-death-rates-from-natural-disasters")
lookup = pd.read_csv(RAW / "iso3166_lookup.csv", dtype=str)
lookup["country-code"] = lookup["country-code"].str.zfill(3)

countries = df[~df["Code"].isna() & ~df["Code"].str.startswith("OWID_")].copy()

merged = countries.merge(
    lookup[["alpha-3", "country-code"]],
    left_on="Code", right_on="alpha-3", how="inner",
)

decades = sorted(merged["Year"].unique().tolist())
countries_out = {}
for _, row in merged.iterrows():
    entry = countries_out.setdefault(row["country-code"], {
        "name": row["Entity"], "iso3": row["Code"], "rates": {},
    })
    rate = row["All disasters"]
    entry["rates"][str(int(row["Year"]))] = None if pd.isna(rate) else round(float(rate), 4)

dropped = countries["Entity"].nunique() - len(countries_out)
write_json("country_death_rate_map.json", {
    "decades": decades,
    "partial_decade": decades[-1] if is_partial_decade(decades[-1]) else None,
    "countries": countries_out,
})
if dropped:
    print(f"note: {dropped} entities had no ISO 3166 match and were left off the map")

wrote data/country_death_rate_map.json
note: 1 entities had no ISO 3166 match and were left off the map


## 5. Source citations

Copied from the OWID metadata files so the methodology section always
matches what was actually downloaded.

In [6]:
files = [
    "natural-disasters-deaths",
    "number-of-natural-disaster-events",
    "decadal-average-death-rates-from-natural-disasters",
]
sources = []
for name in files:
    meta = load_metadata(name)
    chart = meta.get("chart", {})
    sources.append({
        "dataset": name,
        "title": chart.get("title"),
        "note": chart.get("note"),
        "citation": chart.get("citation"),
        "url": chart.get("originalChartUrl"),
        "date_downloaded": meta.get("dateDownloaded"),
    })
write_json("sources.json", sources)

wrote data/sources.json


## Quick sanity checks

A few assertions on the outputs so a broken re-run fails loudly here
instead of producing an empty-looking site.

In [7]:
death_rate = json.load(open(OUT / "death_rate_world.json"))
events = json.load(open(OUT / "events_by_decade.json"))
map_data = json.load(open(OUT / "country_death_rate_map.json"))

assert death_rate["decades"][0] == 1900 and len(death_rate["total"]) == len(death_rate["decades"])
assert death_rate["total"][0] > death_rate["total"][-1], "death rate should fall over the century"
assert events["series"]["All disasters"][0] < events["series"]["All disasters"][-2]
assert len(map_data["countries"]) > 150

print("all checks passed")
print(f'death rate {death_rate["total"][0]} (1900s) -> {death_rate["total"][-1]} (2020s)')

all checks passed
death rate 8.9915 (1900s) -> 0.5944 (2020s)
